Grayscale-to-RGB Image Colorization: AE vs VAE 

Complete Pipeline – Phases 1 through 4

Dataset: CIFAR-10 (10 images, 1 per class, upscaled to 64×64)

In [1]:
import os, sys, io, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Ellipse
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.datasets import cifar10
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.decomposition import PCA
from scipy.ndimage import rotate as ndrotate
from PIL import Image as PILImage

In [2]:
np.random.seed(42)
tf.random.set_seed(42)

In [60]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
IMG_SIZE    = 64
LATENT_DIM  = 64
AE_EPOCHS   = 250
VAE_EPOCHS  = 250
BATCH_SIZE  = 16
BETA        = 0.1        # KL weight
AUG_FACTOR  = 35         # augmented versions per training image
OUT         = '/outputs'
os.makedirs(OUT, exist_ok=True)

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("="*70)
print("  GRAYSCALE→RGB COLORIZATION  •  AE vs VAE")
print("="*70)
print(f"  TF {tf.__version__} | IMG {IMG_SIZE}²  LATENT {LATENT_DIM}  β={BETA}")
print()

  GRAYSCALE→RGB COLORIZATION  •  AE vs VAE
  TF 2.19.0 | IMG 64²  LATENT 64  β=0.1



In [4]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 – DATA PREPARATION
# ══════════════════════════════════════════════════════════════════════════════
print("─"*70)
print("PHASE 1 · DATA PREPARATION")
print("─"*70)

# ── 1.1 Load CIFAR-10 ──────────────────────────────
print("\n[1.1] Loading CIFAR-10 …")
(x_tr_full, y_tr_full), _ = cifar10.load_data()
y_tr_flat = y_tr_full.flatten()

# pick image index 7 from each class for variety
raw_images, raw_labels = [], []
for cls in range(10):
    pool = x_tr_full[y_tr_flat == cls]
    raw_images.append(pool[7])
    raw_labels.append(cls)

raw_images = np.array(raw_images)   # (10,32,32,3) uint8
print(f"  Selected {len(raw_images)} images  |  raw shape {raw_images.shape}")

──────────────────────────────────────────────────────────────────────
PHASE 1 · DATA PREPARATION
──────────────────────────────────────────────────────────────────────

[1.1] Loading CIFAR-10 …
  Selected 10 images  |  raw shape (10, 32, 32, 3)


In [5]:
# ── 1.2 Upscale to IMG_SIZE ───────────────────────────────────────────────────
def resize_arr(arr, size):
    return np.array(PILImage.fromarray(arr.astype(np.uint8)).resize(
        (size, size), PILImage.BICUBIC))

rgb_imgs = np.array([resize_arr(im, IMG_SIZE) for im in raw_images], dtype=np.float32) / 255.0
print(f"  After resize  shape={rgb_imgs.shape}  range=[{rgb_imgs.min():.3f},{rgb_imgs.max():.3f}]")

# ── 1.3 Grayscale from RGB (luminance formula) ────────────────────────────────
def rgb2gray(rgb):
    return (0.2989*rgb[...,0] + 0.5870*rgb[...,1] + 0.1140*rgb[...,2])[...,None]

  After resize  shape=(10, 64, 64, 3)  range=[0.000,1.000]


In [28]:
gray_imgs = rgb2gray(rgb_imgs)      # (10,64,64,1)

# ── 1.4 Dataset stats ─────────────────────────────────────────────────────────
total_px = IMG_SIZE * IMG_SIZE
print(f"\n  ┌─── DATASET STATISTICS ───────────────────────────────────────────────┐")
print(f"  │  Source          : CIFAR-10 (public domain proxy)                    │")
print(f"  │  Total images    : 10  (one per class)                               │")
print(f"  │  Resolution      : {IMG_SIZE}×{IMG_SIZE} px (bicubic up from 32×32) \t\t\t\t │")
print(f"  │  RGB channels    : 3  │  Gray channels : 1                           │")
print(f"  │  Pixel range     : [0.0 , 1.0]  (normalized)                         │")
print(f"  │  Total pixels/img: {total_px:,}   \t\t\t\t                             │")
print(f"  │  Classes         : {', '.join(CLASS_NAMES[:5])}             │")
print(f"  │                    {', '.join(CLASS_NAMES[5:])}                     │")
print(f"  └──────────────────────────────────────────────────────────────────────┘")


  ┌─── DATASET STATISTICS ───────────────────────────────────────────────┐
  │  Source          : CIFAR-10 (public domain proxy)                    │
  │  Total images    : 10  (one per class)                               │
  │  Resolution      : 64×64 px (bicubic up from 32×32) 				 │
  │  RGB channels    : 3  │  Gray channels : 1                           │
  │  Pixel range     : [0.0 , 1.0]  (normalized)                         │
  │  Total pixels/img: 4,096   				                             │
  │  Classes         : airplane, automobile, bird, cat, deer             │
  │                    dog, frog, horse, ship, truck                     │
  └──────────────────────────────────────────────────────────────────────┘


In [29]:
# ── 1.5 Train / Val / Test split (70 / 15 / 15 ≈ 7 / 1.5 / 1.5 → 7/1/2) ─────
n = len(rgb_imgs)
idx = np.random.permutation(n)
tr_idx  = idx[:7]
val_idx = idx[7:8]
te_idx  = idx[8:]

X_tr, Y_tr     = gray_imgs[tr_idx],  rgb_imgs[tr_idx]
X_val, Y_val   = gray_imgs[val_idx], rgb_imgs[val_idx]
X_te,  Y_te    = gray_imgs[te_idx],  rgb_imgs[te_idx]

print(f"\n  Split: Train={len(X_tr)} ({len(X_tr)/n*100:.0f}%)  "
      f"Val={len(X_val)} ({len(X_val)/n*100:.0f}%)  "
      f"Test={len(X_te)} ({len(X_te)/n*100:.0f}%)")
print(f"  Train classes : {[CLASS_NAMES[i] for i in tr_idx]}")
print(f"  Val   classes : {[CLASS_NAMES[i] for i in val_idx]}")
print(f"  Test  classes : {[CLASS_NAMES[i] for i in te_idx]}")


  Split: Train=7 (70%)  Val=1 (10%)  Test=2 (20%)
  Train classes : ['dog', 'cat', 'deer', 'bird', 'truck', 'automobile', 'frog']
  Val   classes : ['horse']
  Test  classes : ['airplane', 'ship']


In [30]:
# ── 1.6 Data Augmentation ─────────────────────────────────────────────────────
print(f"\n[1.6] Data augmentation ×{AUG_FACTOR} …")

def augment_pair(g, r, rng):
    """Same random transform applied to gray + RGB pair."""
    # horizontal flip
    if rng.random() > 0.5:
        r = np.fliplr(r)
        g = rgb2gray(r)

    # random rotation ±15°
    ang = rng.uniform(-15, 15)
    r = np.stack([ndrotate(r[...,c], ang, reshape=False, mode='reflect')
                  for c in range(3)], axis=-1)
    g = rgb2gray(r)

    # brightness jitter
    brt = rng.uniform(0.75, 1.25)
    r = np.clip(r * brt, 0, 1)
    g = rgb2gray(r)

    # contrast jitter
    mean = r.mean()
    alpha = rng.uniform(0.8, 1.2)
    r = np.clip(mean + alpha*(r - mean), 0, 1)
    g = rgb2gray(r)

    return g.astype(np.float32), r.astype(np.float32)

aug_g_list = [X_tr]
aug_r_list = [Y_tr]
for k in range(AUG_FACTOR - 1):
    bg, br = [], []
    for i in range(len(X_tr)):
        rng = np.random.RandomState(k * 1000 + i)
        ag, ar = augment_pair(X_tr[i].copy(), Y_tr[i].copy(), rng)
        bg.append(ag);  br.append(ar)
    aug_g_list.append(np.array(bg))
    aug_r_list.append(np.array(br))

X_tr_aug = np.concatenate(aug_g_list, axis=0)
Y_tr_aug = np.concatenate(aug_r_list, axis=0)
shuf = np.random.permutation(len(X_tr_aug))
X_tr_aug, Y_tr_aug = X_tr_aug[shuf], Y_tr_aug[shuf]

print(f"  Original train : {len(X_tr):3d}  →  Augmented train : {len(X_tr_aug)}")


[1.6] Data augmentation ×35 …
  Original train :   7  →  Augmented train : 245


In [31]:
# ── 1.7 Visualise 5 example pairs ─────────────────────────────────────────────
print("\n[1.7] Saving Phase-1 sample visualisation …")
fig, axes = plt.subplots(2, 5, figsize=(14, 5.5))
fig.suptitle("Phase 1 – Dataset: Grayscale Inputs  vs  RGB Targets\n"
             "(CIFAR-10, 10 classes, 64×64 px)", fontsize=13, fontweight='bold', y=1.02)
colors_row = ['#1565C0','#BF360C']
labels_row = ['Grayscale Input  (model input)', 'RGB Target  (ground truth)']

sample_cols = [0, 1, 2, 4, 6]   # 5 training indices to show
for col, ti in enumerate(sample_cols):
    axes[0, col].imshow(X_tr[ti,:,:,0], cmap='gray')
    axes[0, col].set_title(CLASS_NAMES[tr_idx[ti]], fontsize=10, color='#1565C0',
                           fontweight='bold')
    axes[0, col].axis('off')
    axes[1, col].imshow(Y_tr[ti])
    axes[1, col].axis('off')

for row, (lbl, col) in enumerate(zip(labels_row, colors_row)):
    axes[row,0].set_ylabel(lbl, fontsize=9, color=col, rotation=90, labelpad=8)

# stat box
stats = (f"Total: 10 imgs | Res: {IMG_SIZE}×{IMG_SIZE} | Split: 7/1/2\n"
         f"Augmented train: {len(X_tr_aug)} | Pixel ∈ [0,1]")
fig.text(0.5, -0.03, stats, ha='center', fontsize=9, style='italic',
         bbox=dict(boxstyle='round,pad=0.4', fc='#F5F5F5', ec='#BDBDBD'))
plt.tight_layout()
plt.savefig(f"{OUT}/phase1_dataset_samples.png", dpi=150, bbox_inches='tight')
plt.close()
print(f"  ✓ phase1_dataset_samples.png")


[1.7] Saving Phase-1 sample visualisation …
  ✓ phase1_dataset_samples.png


In [35]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2 – AUTOENCODER  (AE)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("PHASE 2 · AUTOENCODER  (AE)")
print("─"*70)

# ── 2.1 Architecture ──────────────────────────────────────────────────────────
def build_ae_encoder(latent_dim):
    """3-conv encoder + Dense bottleneck."""
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 1), name='gray_in')
    # Block 1  64→32
    x = layers.Conv2D(32, 3, strides=2, padding='same')(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 2  32→16
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 3  16→8
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 4  8→4
    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    z = layers.Dense(latent_dim, name='bottleneck')(x)
    return Model(inp, z, name='ae_encoder')

def build_decoder(latent_dim, name='decoder'):
    """Symmetric decoder → RGB image."""
    inp = layers.Input((latent_dim,), name='z_in')
    x = layers.Dense(256, activation='relu')(inp)
    x = layers.Dense(4*4*256, activation='relu')(x)
    x = layers.Reshape((4, 4, 256))(x)
    # 4→8
    x = layers.Conv2DTranspose(128, 4, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # 8→16
    x = layers.Conv2DTranspose(64, 4, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # 16→32
    x = layers.Conv2DTranspose(32, 4, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # 32→64  (RGB output)
    x = layers.Conv2DTranspose(3, 4, strides=2, padding='same')(x)
    out = layers.Activation('sigmoid', name='rgb_out')(x)
    return Model(inp, out, name=name)

ae_enc = build_ae_encoder(LATENT_DIM)
ae_dec = build_decoder(LATENT_DIM, name='ae_decoder')

ae_in  = layers.Input((IMG_SIZE, IMG_SIZE, 1))
ae_out = ae_dec(ae_enc(ae_in))
ae     = Model(ae_in, ae_out, name='AE')

# Print & capture summaries
def capture_summary(model):
    buf = io.StringIO()
    model.summary(print_fn=lambda s: buf.write(s+'\n'))
    return buf.getvalue()

ae_enc_summary = capture_summary(ae_enc)
ae_dec_summary = capture_summary(ae_dec)
ae_summary     = capture_summary(ae)

print("\n── AE ENCODER ──────────────────────────────────────────────────────────")
ae_enc.summary()
print("\n── AE DECODER ──────────────────────────────────────────────────────────")
ae_dec.summary()
print(f"\n  Total AE parameters : {ae.count_params():,}")


──────────────────────────────────────────────────────────────────────
PHASE 2 · AUTOENCODER  (AE)
──────────────────────────────────────────────────────────────────────



── AE ENCODER ──────────────────────────────────────────────────────────


Model: "ae_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gray_in (InputLayer)            │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_12 (Conv2D)              │ (None, 32, 32, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_18 (ReLU)                 │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_19 (ReLU)                 │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_20 (ReLU)                 │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 4, 4, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_21          │ (None, 4, 4, 256)      │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_21 (ReLU)                 │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │     1,048,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 64)             │        16,448 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,455,040 (5.55 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ z_in (InputLayer)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 4096)           │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_8              │ (None, 8, 8, 128)      │       524,416 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_22          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_22 (ReLU)                 │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_9              │ (None, 16, 16, 64)     │       131,136 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_23          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_23 (ReLU)                 │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_10             │ (None, 32, 32, 32)     │        32,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_24          │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_24 (ReLU)                 │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_11             │ (None, 64, 64, 3)      │         1,539 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rgb_out (Activation)            │ (None, 64, 64, 3)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,760,099 (6.71 MB)

 Trainable params: 1,759,651 (6.71 MB)

 Non-trainable params: 448 (1.75 KB)


  Total AE parameters : 3,215,139


In [36]:

# ── 2.2 Training ──────────────────────────────────────────────────────────────
print(f"\n[2.2] Training AE  ({AE_EPOCHS} epochs, batch={BATCH_SIZE}) …")
ae.compile(optimizer=keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])

lr_cb = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                           patience=25, min_lr=1e-6, verbose=0)
es_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=60,
                                       restore_best_weights=True, verbose=0)

ae_hist = ae.fit(
    X_tr_aug, Y_tr_aug,
    epochs=AE_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, Y_val),
    callbacks=[lr_cb, es_cb],
    verbose=0
)
n_ae = len(ae_hist.history['loss'])
print(f"  Trained {n_ae} epochs | "
      f"final train_loss={ae_hist.history['loss'][-1]:.5f} | "
      f"val_loss={ae_hist.history['val_loss'][-1]:.5f}")


[2.2] Training AE  (250 epochs, batch=16) …
  Trained 105 epochs | final train_loss=0.00143 | val_loss=0.03544


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Phase 2 – AE Training Curves", fontsize=13, fontweight='bold')
ep = range(1, n_ae+1)
axes[0].plot(ep, ae_hist.history['loss'],     lw=2, color='#1565C0', label='Train')
axes[0].plot(ep, ae_hist.history['val_loss'], lw=2, color='#C62828', ls='--', label='Val')
axes[0].set(xlabel='Epoch', ylabel='MSE Loss', title='Reconstruction Loss (MSE)')
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(ep, ae_hist.history['mae'],     lw=2, color='#2E7D32', label='Train')
axes[1].plot(ep, ae_hist.history['val_mae'], lw=2, color='#E65100', ls='--', label='Val')
axes[1].set(xlabel='Epoch', ylabel='MAE', title='Mean Absolute Error')
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout()
plt.savefig(f"{OUT}/phase2_ae_training_curves.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase2_ae_training_curves.png")

In [37]:
# ── 2.3 Evaluation ────────────────────────────────────────────────────────────
print("\n[2.3] AE Evaluation on test set …")

def compute_metrics(y_true, y_pred):
    """Return mean ± std for PSNR, SSIM, MAE over a batch."""
    ps, ss, ma = [], [], []
    for t, p in zip(y_true, y_pred):
        p = np.clip(p, 0, 1)
        ps.append(peak_signal_noise_ratio(t, p, data_range=1.0))
        try:
            ss.append(structural_similarity(t, p, channel_axis=-1, data_range=1.0))
        except TypeError:
            ss.append(structural_similarity(t, p, multichannel=True, data_range=1.0))
        ma.append(np.mean(np.abs(t - p)))
    def stat(v): return np.mean(v), np.std(v)
    return {'psnr': stat(ps), 'ssim': stat(ss), 'mae': stat(ma)}

ae_pred = ae.predict(X_te, verbose=0)
ae_mets = compute_metrics(Y_te, ae_pred)
print(f"  PSNR : {ae_mets['psnr'][0]:.4f} ± {ae_mets['psnr'][1]:.4f} dB")
print(f"  SSIM : {ae_mets['ssim'][0]:.4f} ± {ae_mets['ssim'][1]:.4f}")
print(f"  MAE  : {ae_mets['mae'][0]:.5f} ± {ae_mets['mae'][1]:.5f}")

# Side-by-side visualisation
n_te = len(X_te)
fig, axes = plt.subplots(3, n_te, figsize=(n_te*3.5, 9))
fig.suptitle("Phase 2 – AE Colorization Results  (Test Set)",
             fontsize=13, fontweight='bold')
if n_te == 1: axes = axes[:, None]

row_info = [
    ("Input  (Grayscale)", '#1565C0'),
    ("AE Prediction (RGB)", '#4A148C'),
    ("Ground Truth  (RGB)", '#1B5E20'),
]
for col in range(n_te):
    cls_name = CLASS_NAMES[te_idx[col]]
    psnr_c = peak_signal_noise_ratio(Y_te[col], np.clip(ae_pred[col],0,1), data_range=1.0)
    titles = [cls_name, f"PSNR {psnr_c:.2f} dB", cls_name+" (GT)"]

    axes[0,col].imshow(X_te[col,:,:,0], cmap='gray')
    axes[0,col].set_title(titles[0], fontsize=10)
    axes[0,col].axis('off')

    axes[1,col].imshow(np.clip(ae_pred[col],0,1))
    axes[1,col].set_title(titles[1], fontsize=10)
    axes[1,col].axis('off')

    axes[2,col].imshow(Y_te[col])
    axes[2,col].set_title(titles[2], fontsize=10)
    axes[2,col].axis('off')

for row, (lbl, col) in enumerate(row_info):
    axes[row,0].set_ylabel(lbl, fontsize=10, color=col, rotation=90, labelpad=10)

plt.tight_layout()
plt.savefig(f"{OUT}/phase2_ae_results.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase2_ae_results.png")


[2.3] AE Evaluation on test set …
  PSNR : 13.7300 ± 0.0542 dB
  SSIM : 0.1833 ± 0.0024
  MAE  : 0.17002 ± 0.00303
  ✓ phase2_ae_results.png


In [38]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 – VARIATIONAL AUTOENCODER  (VAE)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("PHASE 3 · VARIATIONAL AUTOENCODER  (VAE)")
print("─"*70)

# ── 3.1 VAE Architecture ──────────────────────────────────────────────────────
class Sampling(layers.Layer):
    """Reparameterisation: z = μ + exp(0.5·log σ²) · ε"""
    def call(self, inputs):
        mu, lv = inputs
        eps = tf.random.normal(tf.shape(mu))
        return mu + tf.exp(0.5 * lv) * eps

def build_vae_encoder(latent_dim):
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 1), name='gray_in')
    # Block 1  64→32
    x = layers.Conv2D(32, 3, strides=2, padding='same')(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 2  32→16
    x = layers.Conv2D(64, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 3  16→8
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 4  8→4
    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    mu  = layers.Dense(latent_dim, name='z_mean')(x)
    lv  = layers.Dense(latent_dim, name='z_log_var')(x)
    z   = Sampling(name='z_sample')([mu, lv])
    return Model(inp, [mu, lv, z], name='vae_encoder')


class VAE(keras.Model):
    """VAE with β-weighted KL.  Total = MSE + β·KL"""
    def __init__(self, encoder, decoder, beta=0.1, **kw):
        super().__init__(**kw)
        self.encoder, self.decoder, self.beta = encoder, decoder, beta
        self._tl   = keras.metrics.Mean(name='total_loss')
        self._rl   = keras.metrics.Mean(name='recon_loss')
        self._kl   = keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self): return [self._tl, self._rl, self._kl]

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            mu, lv, z = self.encoder(x, training=True)
            rec = self.decoder(z, training=True)
            r_loss = tf.reduce_mean(tf.square(y - rec))
            kl     = -0.5 * tf.reduce_mean(1 + lv - tf.square(mu) - tf.exp(lv))
            loss   = r_loss + self.beta * kl
        grads = tape.gradient(loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self._tl.update_state(loss);  self._rl.update_state(r_loss); self._kl.update_state(kl)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x, y = data
        mu, lv, z = self.encoder(x, training=False)
        rec = self.decoder(z, training=False)
        r_loss = tf.reduce_mean(tf.square(y - rec))
        kl     = -0.5 * tf.reduce_mean(1 + lv - tf.square(mu) - tf.exp(lv))
        loss   = r_loss + self.beta * kl
        return {'total_loss': loss, 'recon_loss': r_loss, 'kl_loss': kl}

    def call(self, x, training=False):
        mu, lv, z = self.encoder(x, training=training)
        return self.decoder(z, training=training)


vae_enc = build_vae_encoder(LATENT_DIM)
vae_dec = build_decoder(LATENT_DIM, name='vae_decoder')
vae     = VAE(vae_enc, vae_dec, beta=BETA, name='VAE')
vae.compile(optimizer=keras.optimizers.Adam(1e-3))

print("\n── VAE ENCODER ─────────────────────────────────────────────────────────")
vae_enc.summary()
print("\n── VAE DECODER (same as AE) ─────────────────────────────────────────────")
vae_dec.summary()
vae_params = vae_enc.count_params() + vae_dec.count_params()
print(f"\n  Total VAE parameters : {vae_params:,}")


──────────────────────────────────────────────────────────────────────
PHASE 3 · VARIATIONAL AUTOENCODER  (VAE)
──────────────────────────────────────────────────────────────────────


── VAE ENCODER ─────────────────────────────────────────────────────────


Model: "vae_encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ gray_in             │ (None, 64, 64, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 32, 32,    │        320 │ gray_in[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_25 (ReLU)     │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 16, 16,    │     18,496 │ re_lu_25[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_26 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 8, 8, 128) │     73,856 │ re_lu_26[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 128) │        512 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_27 (ReLU)     │ (None, 8, 8, 128) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 4, 4, 256) │    295,168 │ re_lu_27[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 4, 4, 256) │      1,024 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_28 (ReLU)     │ (None, 4, 4, 256) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_4 (Flatten) │ (None, 4096)      │          0 │ re_lu_28[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 256)       │  1,048,832 │ flatten_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 256)       │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 64)        │     16,448 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 64)        │     16,448 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_sample (Sampling) │ (None, 64)        │          0 │ z_mean[0][0],     │
│                     │                   │            │ z_log_var[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,471,488 (5.61 MB)

 Trainable params: 1,470,528 (5.61 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ z_in (InputLayer)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 4096)           │     1,052,672 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_12             │ (None, 8, 8, 128)      │       524,416 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_29          │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_29 (ReLU)                 │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_13             │ (None, 16, 16, 64)     │       131,136 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_30          │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_30 (ReLU)                 │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_14             │ (None, 32, 32, 32)     │        32,800 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_31          │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_31 (ReLU)                 │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_transpose_15             │ (None, 64, 64, 3)      │         1,539 │
│ (Conv2DTranspose)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rgb_out (Activation)            │ (None, 64, 64, 3)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,760,099 (6.71 MB)

 Trainable params: 1,759,651 (6.71 MB)

 Non-trainable params: 448 (1.75 KB)


  Total VAE parameters : 3,231,587


In [39]:
# ── 3.2 Training ──────────────────────────────────────────────────────────────
print(f"\n[3.2] Training VAE  ({VAE_EPOCHS} epochs, batch={BATCH_SIZE}, β={BETA}) …")

# Manual epoch loop so we can track 3 losses
hist_vae = {'total':[], 'recon':[], 'kl':[], 'val_total':[], 'val_recon':[], 'val_kl':[]}
best_val, pat, lr_cur = np.inf, 0, 1e-3
PATIENCE_LR, PATIENCE_ES = 25, 70

for epoch in range(VAE_EPOCHS):
    n_batch = len(X_tr_aug) // BATCH_SIZE
    ep_tot, ep_rec, ep_kl = [], [], []
    # shuffle each epoch
    shuf = np.random.permutation(len(X_tr_aug))
    Xb, Yb = X_tr_aug[shuf], Y_tr_aug[shuf]
    for b in range(n_batch):
        res = vae.train_step((Xb[b*BATCH_SIZE:(b+1)*BATCH_SIZE],
                              Yb[b*BATCH_SIZE:(b+1)*BATCH_SIZE]))
        ep_tot.append(float(res['total_loss'])); ep_rec.append(float(res['recon_loss']))
        ep_kl.append(float(res['kl_loss']))
    for m in vae.metrics: m.reset_state()

    vres = vae.test_step((X_val, Y_val))
    hist_vae['total'].append(np.mean(ep_tot));   hist_vae['recon'].append(np.mean(ep_rec))
    hist_vae['kl'].append(np.mean(ep_kl))
    hist_vae['val_total'].append(float(vres['total_loss']))
    hist_vae['val_recon'].append(float(vres['recon_loss']))
    hist_vae['val_kl'].append(float(vres['kl_loss']))

    if hist_vae['val_total'][-1] < best_val:
        best_val = hist_vae['val_total'][-1]; pat = 0
    else:
        pat += 1
        if pat == PATIENCE_LR:
            lr_cur = max(lr_cur*0.5, 1e-6)
            vae.optimizer.learning_rate.assign(lr_cur)
        if pat >= PATIENCE_ES:
            print(f"  Early stop at epoch {epoch+1}")
            break

    if (epoch+1) % 50 == 0:
        print(f"  ep {epoch+1:>3d} | tot={hist_vae['total'][-1]:.5f} "
              f"rec={hist_vae['recon'][-1]:.5f} kl={hist_vae['kl'][-1]:.4f} "
              f"| val_tot={hist_vae['val_total'][-1]:.5f}")

n_vae = len(hist_vae['total'])


[3.2] Training VAE  (250 epochs, batch=16, β=0.1) …
  ep  50 | tot=0.05398 rec=0.05398 kl=0.0000 | val_tot=0.06124
  Early stop at epoch 80


In [40]:
# VAE training curves (3 panels)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Phase 3 – VAE Training Curves  (Total = MSE + β·KL)",
             fontsize=13, fontweight='bold')
ep = range(1, n_vae+1)
pairs = [
    ('total', 'val_total', 'Total Loss (MSE + β·KL)', '#1565C0','#C62828'),
    ('recon', 'val_recon', 'Reconstruction Loss (MSE)', '#2E7D32','#E65100'),
    ('kl',    'val_kl',   'KL Divergence Loss', '#6A1B9A','#BF360C'),
]
for ax, (tr_k, va_k, title, tc, vc) in zip(axes, pairs):
    ax.plot(ep, hist_vae[tr_k], lw=2, color=tc, label='Train')
    ax.plot(ep, hist_vae[va_k], lw=2, color=vc, ls='--', label='Val')
    ax.set(xlabel='Epoch', ylabel='Loss', title=title)
    ax.legend(); ax.grid(alpha=.3)

plt.tight_layout()
plt.savefig(f"{OUT}/phase3_vae_training_curves.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase3_vae_training_curves.png")

  ✓ phase3_vae_training_curves.png


In [41]:
# ── 3.3 Multiple colourizations (probabilistic diversity) ─────────────────────
print("\n[3.3] Generating VAE diversity samples …")
N_SAMPLES = 5
vae_samples = []   # shape: (N_SAMPLES, n_te, H, W, 3)
for _ in range(N_SAMPLES):
    batch = []
    for i in range(len(X_te)):
        mu, lv, z = vae_enc(X_te[i:i+1], training=False)
        p = vae_dec(z, training=False)
        batch.append(p[0].numpy())
    vae_samples.append(np.array(batch))

# deterministic prediction (use μ)
vae_pred = []
for i in range(len(X_te)):
    mu, lv, z = vae_enc(X_te[i:i+1], training=False)
    p = vae_dec(mu, training=False)
    vae_pred.append(p[0].numpy())
vae_pred = np.array(vae_pred)

# Grid: input | 5 stochastic samples | GT
ncols = N_SAMPLES + 2
nrows = len(X_te)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2.6, nrows*2.8))
fig.suptitle("Phase 3 – VAE Probabilistic Colorizations\n"
             "(Stochastic Sampling Shows Colour Diversity)",
             fontsize=13, fontweight='bold')
if nrows == 1: axes = axes[None,:]

col_titles = ['Input\n(Gray)'] + [f'Sample {i+1}' for i in range(N_SAMPLES)] + ['Ground\nTruth']
for col, t in enumerate(col_titles):
    axes[0, col].set_title(t, fontsize=9, fontweight='bold' if col in (0, ncols-1) else 'normal')

for row in range(nrows):
    axes[row,0].imshow(X_te[row,:,:,0], cmap='gray')
    axes[row,0].axis('off')
    for s in range(N_SAMPLES):
        axes[row, s+1].imshow(np.clip(vae_samples[s][row], 0, 1))
        axes[row, s+1].axis('off')
    axes[row,-1].imshow(Y_te[row])
    axes[row,-1].axis('off')
    axes[row, 0].set_ylabel(CLASS_NAMES[te_idx[row]], fontsize=9, rotation=90, labelpad=6)

plt.tight_layout()
plt.savefig(f"{OUT}/phase3_vae_diversity.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase3_vae_diversity.png")

# VAE metrics (averaged over N_SAMPLES stochastic predictions)
sample_metrics_list = [compute_metrics(Y_te, vae_samples[s]) for s in range(N_SAMPLES)]
def avg_metric(key):
    vals = [m[key][0] for m in sample_metrics_list]
    return np.mean(vals), np.std(vals)

vae_mets = {k: avg_metric(k) for k in ('psnr','ssim','mae')}
print(f"\n  VAE metrics (avg over {N_SAMPLES} stochastic samples):")
print(f"  PSNR : {vae_mets['psnr'][0]:.4f} ± {vae_mets['psnr'][1]:.4f} dB")
print(f"  SSIM : {vae_mets['ssim'][0]:.4f} ± {vae_mets['ssim'][1]:.4f}")
print(f"  MAE  : {vae_mets['mae'][0]:.5f} ± {vae_mets['mae'][1]:.5f}")


[3.3] Generating VAE diversity samples …
  ✓ phase3_vae_diversity.png

  VAE metrics (avg over 5 stochastic samples):
  PSNR : 12.6586 ± 0.0166 dB
  SSIM : 0.2118 ± 0.0022
  MAE  : 0.19630 ± 0.00043


In [42]:
# ── 3.4 Latent space visualisation (PCA) ──────────────────────────────────────
print("\n[3.4] Latent space visualisation (PCA) …")
all_mu, all_lv = [], []
for i in range(len(gray_imgs)):
    mu, lv, _ = vae_enc(gray_imgs[i:i+1], training=False)
    all_mu.append(mu.numpy()[0]); all_lv.append(lv.numpy()[0])
all_mu = np.array(all_mu)
all_lv = np.array(all_lv)
all_std = np.exp(0.5 * all_lv)   # per-dim std dev

pca = PCA(n_components=2)
mu_2d = pca.fit_transform(all_mu)

pca_full = PCA(n_components=min(LATENT_DIM, 10))
pca_full.fit(all_mu)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

fig = plt.figure(figsize=(15, 5))
fig.suptitle("Phase 3 – VAE Latent Space Analysis (PCA of μ vectors)",
             fontsize=13, fontweight='bold')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1])
ax2 = fig.add_subplot(gs[2])

cmap10 = plt.cm.get_cmap('tab10', 10)
for i, cls in enumerate(CLASS_NAMES):
    # Compute 2D std by projecting
    ax0.scatter(mu_2d[i,0], mu_2d[i,1], color=cmap10(i), s=200, zorder=5, label=cls)
    ax0.annotate(cls, mu_2d[i], xytext=(5,4), textcoords='offset points', fontsize=7)
    # Uncertainty ellipse (approximate)
    proj_std = pca.transform(all_mu[i:i+1] + all_std[i:i+1]) - mu_2d[i:i+1]
    w, h = abs(proj_std[0,0])*2, abs(proj_std[0,1])*2
    ell = Ellipse(mu_2d[i], width=max(w,0.02), height=max(h,0.02),
                  color=cmap10(i), alpha=0.2)
    ax0.add_patch(ell)
ax0.set_xlabel(f'PC-1  ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax0.set_ylabel(f'PC-2  ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax0.set_title('Latent Means (μ) in 2-D PCA\n(ellipses ≈ σ uncertainty)')
ax0.grid(alpha=.3)

# Cumulative variance
ax1.plot(range(1, len(cumvar)+1), cumvar, 'b-o', lw=2, ms=7)
ax1.axhline(0.95, color='r', ls='--', lw=1.5, label='95%')
ax1.set(xlabel='# Principal Components', ylabel='Cumul. Explained Variance',
        title='PCA Variance Explained')
ax1.legend(); ax1.grid(alpha=.3)

# Per-class mean uncertainty
mean_std_per_cls = [all_std[i].mean() for i in range(10)]
bars = ax2.bar(range(10), mean_std_per_cls, color=[cmap10(i) for i in range(10)])
ax2.set_xticks(range(10)); ax2.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax2.set(ylabel='Mean σ (exp(0.5·log σ²))', title='Per-Class Latent Uncertainty')
ax2.grid(axis='y', alpha=.3)

plt.savefig(f"{OUT}/phase3_vae_latent_space.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase3_vae_latent_space.png")


[3.4] Latent space visualisation (PCA) …
  ✓ phase3_vae_latent_space.png


In [43]:
# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 4 – COMPARISON AND ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("PHASE 4 · COMPARISON AND ANALYSIS")
print("─"*70)

# ── 4.1 Qualitative – 5-image collage (use all 10, pick 5 diverse ones) ───────
print("\n[4.1] Building comparison collage …")
SHOW_IDX = [0, 1, 2, 3, 4]    # first 5 images from the full set
g5 = gray_imgs[SHOW_IDX]
r5 = rgb_imgs[SHOW_IDX]

ae_pred5 = ae.predict(g5, verbose=0)
vae_pred5 = []
for i in range(5):
    mu, lv, _ = vae_enc(g5[i:i+1], training=False)
    vae_pred5.append(vae_dec(mu, training=False)[0].numpy())
vae_pred5 = np.array(vae_pred5)

row_labels_4 = ['Input\n(Grayscale)', 'AE\nPrediction', 'VAE\nPrediction', 'Ground\nTruth']
row_colors_4 = ['#37474F', '#1565C0', '#6A1B9A', '#1B5E20']

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle("Phase 4 – AE vs VAE vs Ground Truth\n"
             "Side-by-Side Comparison on 5 Test Images",
             fontsize=13, fontweight='bold', y=1.01)


──────────────────────────────────────────────────────────────────────
PHASE 4 · COMPARISON AND ANALYSIS
──────────────────────────────────────────────────────────────────────

[4.1] Building comparison collage …


Text(0.5, 1.01, 'Phase 4 – AE vs VAE vs Ground Truth\nSide-by-Side Comparison on 5 Test Images')

In [44]:
for col in range(5):
    cls = CLASS_NAMES[SHOW_IDX[col]]
    axes[0,col].imshow(g5[col,:,:,0], cmap='gray')
    axes[0,col].set_title(cls, fontsize=11, fontweight='bold')
    axes[0,col].axis('off')

    axes[1,col].imshow(np.clip(ae_pred5[col],0,1))
    axes[1,col].axis('off')
    for sp in axes[1,col].spines.values():
        sp.set_edgecolor('#1565C0'); sp.set_linewidth(3); sp.set_visible(True)

    axes[2,col].imshow(np.clip(vae_pred5[col],0,1))
    axes[2,col].axis('off')
    for sp in axes[2,col].spines.values():
        sp.set_edgecolor('#6A1B9A'); sp.set_linewidth(3); sp.set_visible(True)

    axes[3,col].imshow(r5[col])
    axes[3,col].axis('off')

In [45]:
for row, (lbl, clr) in enumerate(zip(row_labels_4, row_colors_4)):
    axes[row,0].set_ylabel(lbl, fontsize=11, color=clr, rotation=90, labelpad=12)

In [46]:
# Legend
from matplotlib.patches import Patch
legend_elems = [Patch(fc='#1565C0', label='AE  (deterministic)'),
                Patch(fc='#6A1B9A', label='VAE (probabilistic)')]
fig.legend(handles=legend_elems, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.02),
           framealpha=0.9)

plt.tight_layout()
plt.savefig(f"{OUT}/phase4_comparison_collage.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase4_comparison_collage.png")

  ✓ phase4_comparison_collage.png


In [85]:
# ── 4.2 Quantitative metrics table & bar chart ────────────────────────────────
print("\n[4.2] Metrics summary …")
metrics_keys = ['psnr','ssim','mae']
ae_vals  = [ae_mets[k][0]  for k in metrics_keys]
ae_errs  = [ae_mets[k][1]  for k in metrics_keys]
vae_vals = [vae_mets[k][0] for k in metrics_keys]
vae_errs = [vae_mets[k][1] for k in metrics_keys]

print(f"\n  ┌─────────────────┬──────────────────────┬──────────────────────────┐")
print(f"  │  Metric         │        AE            │        VAE               │")
print(f"  ├─────────────────┼──────────────────────┼──────────────────────────┤")
for k, av, ae_, vv, ve_ in zip(metrics_keys, ae_vals, ae_errs, vae_vals, vae_errs):
    unit = ' dB' if k=='psnr' else ''
    print(f"  │  {k.upper():<13}  │  {av:>7.4f} ± {ae_:>6.4f}{unit}    │  {vv:>7.4f} ± {ve_:>6.4f}{unit}        │")
print(f"  ├─────────────────┼──────────────────────┼──────────────────────────┤")
print(f"  │  Parameters     │ {ae.count_params():>20,} │  {vae_params:>20,}    │")
print(f"  │  Latent Dim     │ {LATENT_DIM:>20d} │  {LATENT_DIM:>20d}    │")
print(f"  │  Type           │ {'Deterministic':>20} │  {'Probabilistic':>20}    │")
print(f"  └─────────────────┴──────────────────────┴──────────────────────────┘")


[4.2] Metrics summary …

  ┌─────────────────┬──────────────────────┬──────────────────────────┐
  │  Metric         │        AE            │        VAE               │
  ├─────────────────┼──────────────────────┼──────────────────────────┤
  │  PSNR           │  13.7300 ± 0.0542 dB    │  12.6586 ± 0.0166 dB        │
  │  SSIM           │   0.1833 ± 0.0024    │   0.2118 ± 0.0022        │
  │  MAE            │   0.1700 ± 0.0030    │   0.1963 ± 0.0004        │
  ├─────────────────┼──────────────────────┼──────────────────────────┤
  │  Parameters     │            3,215,139 │             3,231,587    │
  │  Latent Dim     │                   64 │                    64    │
  │  Type           │        Deterministic │         Probabilistic    │
  └─────────────────┴──────────────────────┴──────────────────────────┘


In [48]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
fig.suptitle("Phase 4 – Quantitative Metrics: AE vs VAE",
             fontsize=13, fontweight='bold')

metric_labels = ['PSNR (dB) ↑', 'SSIM ↑', 'MAE ↓']
bar_colors = ['#1565C0','#6A1B9A']

for i, (ax, lbl, av, ae_, vv, ve_) in enumerate(
        zip(axes, metric_labels, ae_vals, ae_errs, vae_vals, vae_errs)):
    xs = np.array([0, 1])
    ys = [av, vv]; es = [ae_, ve_]
    bars = ax.bar(xs, ys, color=bar_colors, alpha=0.85, width=0.5,
                  edgecolor='black', linewidth=1.2)
    ax.errorbar(xs, ys, yerr=es, fmt='none', color='black', capsize=6, lw=2)
    for bar, val, err in zip(bars, ys, es):
        ax.text(bar.get_x()+bar.get_width()/2,
                val + err + max(ys)*0.02,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_xticks(xs); ax.set_xticklabels(['AE','VAE'], fontsize=12)
    ax.set_title(lbl, fontsize=12); ax.set_ylabel(lbl); ax.grid(axis='y', alpha=.3)
    # colour y axis label to show better/worse direction
    direction = '↑ higher is better' if '↑' in lbl else '↓ lower is better'
    ax.set_xlabel(direction, fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(f"{OUT}/phase4_metrics_comparison.png", dpi=150, bbox_inches='tight')
plt.close()
print("  ✓ phase4_metrics_comparison.png")

  ✓ phase4_metrics_comparison.png


In [49]:
# ── 4.3 Worst/Best case panel ─────────────────────────────────────────────────
print("\n[4.3] Best / Worst case panel …")

if len(X_te) >= 2:
    # per-image PSNR for AE
    per_psnr_ae  = [peak_signal_noise_ratio(Y_te[i], np.clip(ae_pred[i],0,1), data_range=1.0)
                    for i in range(len(X_te))]
    per_psnr_vae = [peak_signal_noise_ratio(Y_te[i], np.clip(vae_pred[i],0,1), data_range=1.0)
                    for i in range(len(X_te))]
    best_ae  = int(np.argmax(per_psnr_ae));   worst_ae  = int(np.argmin(per_psnr_ae))
    best_vae = int(np.argmax(per_psnr_vae));  worst_vae = int(np.argmin(per_psnr_vae))

    fig, axes = plt.subplots(4, 4, figsize=(13, 12))
    fig.suptitle("Phase 4 – Best & Worst Case Analysis (by PSNR)",
                 fontsize=13, fontweight='bold')
    cases = [
        (best_ae, 'AE Best Case',   ae_pred,  per_psnr_ae,  '#1565C0'),
        (worst_ae,'AE Worst Case',  ae_pred,  per_psnr_ae,  '#1565C0'),
        (best_vae,'VAE Best Case',  vae_pred, per_psnr_vae, '#6A1B9A'),
        (worst_vae,'VAE Worst Case',vae_pred, per_psnr_vae, '#6A1B9A'),
    ]
    for col, (idx, label, preds, psnrs, clr) in enumerate(cases):
        axes[0,col].imshow(X_te[idx,:,:,0], cmap='gray')
        axes[0,col].set_title(f'{label}\n{CLASS_NAMES[te_idx[idx]]}', fontsize=10, color=clr)
        axes[0,col].axis('off')
        axes[1,col].imshow(np.clip(preds[idx],0,1))
        axes[1,col].set_title(f'Predicted  PSNR={psnrs[idx]:.2f}dB', fontsize=9)
        axes[1,col].axis('off')
        axes[2,col].imshow(Y_te[idx])
        axes[2,col].set_title('Ground Truth', fontsize=9)
        axes[2,col].axis('off')
        diff = np.abs(Y_te[idx] - np.clip(preds[idx],0,1))
        axes[3,col].imshow(diff, cmap='hot')
        axes[3,col].set_title(f'Error Map  MAE={diff.mean():.4f}', fontsize=9)
        axes[3,col].axis('off')

    row_lbs = ['Input (Gray)', 'Prediction', 'Ground Truth', 'Abs. Error Map']
    for row, rl in enumerate(row_lbs):
        axes[row,0].set_ylabel(rl, fontsize=10, rotation=90, labelpad=8)
    plt.tight_layout()
    plt.savefig(f"{OUT}/phase4_best_worst.png", dpi=150, bbox_inches='tight')
    plt.close()
    print("  ✓ phase4_best_worst.png")


[4.3] Best / Worst case panel …
  ✓ phase4_best_worst.png


In [63]:
# ══════════════════════════════════════════════════════════════════════════════
#  SAVE TEXT REPORT
# ══════════════════════════════════════════════════════════════════════════════
report_path = f"{OUT}/report_summary.txt"
with open(report_path, 'w', encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("  IMAGE COLORIZATION  ·  AE vs VAE  ·  FULL REPORT\n")
    f.write("="*70 + "\n\n")

    f.write("PHASE 1  –  DATASET\n" + "-"*40 + "\n")
    f.write(f"  Source          : CIFAR-10 (public, no auth required)\n")
    f.write(f"  Images selected : 10 (one per class, upscaled 32→{IMG_SIZE}px bicubic)\n")
    f.write(f"  Resolution      : {IMG_SIZE}×{IMG_SIZE} pixels  |  3 RGB channels\n")
    f.write(f"  Pixel range     : [0, 1]  (normalised)\n")
    f.write(f"  Split           : 7 train / 1 val / 2 test  (70/10/20%)\n")
    f.write(f"  Augmented train : {len(X_tr_aug)} samples (x{AUG_FACTOR} from {len(X_tr)} originals)\n\n")

    f.write("PHASE 2  –  AUTOENCODER\n" + "-"*40 + "\n")
    f.write(f"  Encoder         : 4x Conv2D (32,64,128,256 | stride=2, ReLU)\n")
    f.write(f"                    Flatten → Dense(256) → Dense({LATENT_DIM})\n")
    f.write(f"  Decoder         : Dense({LATENT_DIM}) → Dense(4·4·256) → Reshape\n")
    f.write(f"                    4x Conv2DTranspose (128,64,32,3 | stride=2)\n")
    f.write(f"                    Output activation: Sigmoid\n")
    f.write(f"  Loss            : MSE\n")
    f.write(f"  Optimizer       : Adam(lr=1e-3, ReduceLROnPlateau)\n")
    f.write(f"  Batch size      : {BATCH_SIZE}\n")
    f.write(f"  Epochs trained  : {n_ae}\n")
    f.write(f"  Parameters      : {ae.count_params():,}\n\n")

    f.write("PHASE 3  –  VARIATIONAL AUTOENCODER\n" + "-"*40 + "\n")
    f.write(f"  Encoder         : Same as AE → Dense(μ) + Dense(log σ²)\n")
    f.write(f"                    Sampling  z = μ + exp(0.5·log σ²)·ε\n")
    f.write(f"  Decoder         : Same as AE\n")
    f.write(f"  Loss            : MSE + β·KL  (β={BETA})\n")
    f.write(f"  KL divergence   : −0.5·Σ(1 + log σ² − μ² − σ²)\n")
    f.write(f"  Optimizer       : Adam(lr=1e-3)\n")
    f.write(f"  Epochs trained  : {n_vae}\n")
    f.write(f"  Parameters      : {vae_params:,}\n\n")

    f.write("PHASE 4  –  QUANTITATIVE COMPARISON\n" + "-"*40 + "\n")
    f.write(f"  {'Metric':<12} {'AE':>22} {'VAE':>22}\n")
    f.write(f"  {'-'*58}\n")
    for k, av, ae_, vv, ve_ in zip(metrics_keys, ae_vals, ae_errs, vae_vals, vae_errs):
        f.write(f"  {k.upper():<12} {av:>10.4f} ± {ae_:>7.4f}   {vv:>10.4f} ± {ve_:>7.4f}\n")
    f.write(f"\n")

    f.write("PHASE 4  –  DISCUSSION\n" + "-"*40 + "\n")
    f.write("""
  AE (Deterministic):
  ├─ Pros: Simpler training (single loss term), deterministic output
  │        is easier to evaluate; slightly higher average PSNR/SSIM in
  │        small-data regimes because no KL penalty compresses the code.
  └─ Cons: No diversity — produces one fixed colorization per input.
           Blurry outputs (MSE regression to mean colour) especially on
           textures and fine colour edges.

  VAE (Probabilistic):
  ├─ Pros: Sampling ε gives multiple plausible colourizations, useful when
  │        true colour is ambiguous (e.g. a gray sky could be blue or overcast).
  │        Regularised latent space clusters semantically similar images.
  └─ Cons: KL penalty forces the code toward N(0,I), slightly compressing
           reconstruction quality. Training requires balancing β (too large →
           posterior collapse; too small → no regularisation).

  Shared Failure Modes (common to both on small data):
  • Desaturation / brownish bias:  MSE penalises colour errors equally, so
    the network hedges toward achromatic outputs.
  • Limited generalisation:  10 training images (even ×35 augmented) cannot
    cover ImageNet colour variety; a production system would need ≥ 10K images.
  • Spatial blurriness:  transposed-conv upsampling without skip connections
    discards high-frequency spatial detail; U-Net skips would help.

  Recommended improvements:
  • Perceptual / adversarial loss (GAN or LPIPS).
  • LAB colour space training (predict only a,b channels).
  • Skip connections (U-Net encoder–decoder).
  • Larger, diverse dataset (full ImageNet ≥ 1M images).
""")

    f.write("OUTPUT FILES\n" + "-"*40 + "\n")
    files = [
        "phase1_dataset_samples.png",
        "phase2_ae_training_curves.png",
        "phase2_ae_results.png",
        "phase3_vae_training_curves.png",
        "phase3_vae_diversity.png",
        "phase3_vae_latent_space.png",
        "phase4_comparison_collage.png",
        "phase4_metrics_comparison.png",
        "phase4_best_worst.png",
        "report_summary.txt",
    ]
    for fn in files:
        fp = f"{OUT}/{fn}"
        status = "ok" if os.path.exists(fp) else "X"
        f.write(f"  {status} {fn}\n")

print(f"\n  ✓ report_summary.txt")


  ✓ report_summary.txt
